# 🔬 Comparación de Modelos

En este notebook entrenamos dos modelos tradicionales basados en árboles de decisión:
1. **Random Forest Classifier**
2. **XGBoost Classifier**

Y los comparamos con el desempeño obtenido por nuestra Red Neuronal Multicapa (MLP) utilizando métricas estándar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score

sns.set_theme(style="whitegrid")

### 1. Carga de Datos y Predicciones de MLP

In [ ]:
# Datos listos para modelar
data = np.load('../data/processed/model_ready_data.npz')
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']

# Predicciones del modelo MLP
mlp_data = np.load('../data/processed/mlp_predictions.npz')
y_pred_prob_mlp = mlp_data['y_pred_prob']
y_pred_mlp = mlp_data['y_pred']

print("Datos cargados correctamente.")

### 2. Entrenamiento de Modelos Comparativos

In [ ]:
# 2.1. Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred_prob_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = rf.predict(X_test)

print("--- Random Forest Report ---")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# 2.2. XGBoost Classifier
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred_prob_xgb = xgb.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb.predict(X_test)

print("--- XGBoost Report ---")
print(classification_report(y_test, y_pred_xgb))

### 3. Comparación de Curvas ROC

In [ ]:
# Curva ROC para MLP
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_pred_prob_mlp)
auc_mlp = auc(fpr_mlp, tpr_mlp)

# Curva ROC para Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_prob_rf)
auc_rf = auc(fpr_rf, tpr_rf)

# Curva ROC para XGBoost
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_prob_xgb)
auc_xgb = auc(fpr_xgb, tpr_xgb)

plt.figure(figsize=(10, 8))
plt.plot(fpr_mlp, tpr_mlp, label=f'Red Neuronal (MLP) (AUC = {auc_mlp:.4f})', color='darkred', lw=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.4f})', color='forestgreen', lw=2)
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {auc_xgb:.4f})', color='darkblue', lw=2)
plt.plot([0, 1], [0, 1], color='grey', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
plt.title('Comparación de Curvas ROC de los Modelos Evaluados', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.show()

### 4. Tabla Comparativa de Métricas

In [ ]:
model_names = ['Red Neuronal (MLP)', 'Random Forest', 'XGBoost']
preds = [y_pred_mlp, y_pred_rf, y_pred_xgb]
probs = [y_pred_prob_mlp, y_pred_prob_rf, y_pred_prob_xgb]

metrics_data = []
for name, pred, prob in zip(model_names, preds, probs):
    fpr, tpr, _ = roc_curve(y_test, prob)
    metrics_data.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision (Stroke)': precision_score(y_test, pred),
        'Recall (Stroke)': recall_score(y_test, pred),
        'F1-Score (Stroke)': f1_score(y_test, pred),
        'ROC-AUC': auc(fpr, tpr)
    })

df_comparison = pd.DataFrame(metrics_data)
df_comparison

### Conclusión Comparativa

- **Recall:** En la predicción de stroke (un caso médico), el **Recall** (Sensibilidad) es la métrica más importante, dado que nos interesa minimizar los Falsos Negativos (pacientes en riesgo que el modelo no detecta).
- **SMOTE:** La aplicación de SMOTE ayudó a mejorar significativamente el recall de todos los modelos.
- **MLP vs Árboles:** La Red Neuronal Artificial (MLP) ofrece un desempeño balanceado y ajustable en el umbral de decisión, logrando una frontera de decisión no lineal robusta frente al desbalance estructural de los datos.